Copyright 2025 Google LLC<br>
<br>
Licensed under the Apache License, Version 2.0 (the "License");<br>
you may not use this file except in compliance with the License.<br>
You may obtain a copy of the License at<br>
<br>
    http://www.apache.org/licenses/LICENSE-2.0<br>
<br>
Unless required by applicable law or agreed to in writing, software<br>
distributed under the License is distributed on an "AS IS" BASIS,<br>
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.<br>
See the License for the specific language governing permissions and<br>
limitations under the License.


Reviser agent for correcting inaccuracies based on verified findings.
<br>
import os<br>
from google.adk import Agent<br>
from google.adk.agents.callback_context import CallbackContext<br>
from google.adk.models import LlmResponse<br>
import sys<br>
sys.path.append("../..")<br>
from callback_logging import log_query_to_model, log_model_response<br>
from . import prompt<br>
_END_OF_EDIT_MARK = '---END-OF-EDIT---'<br>
def _remove_end_of_edit_mark(<br>
    callback_context: CallbackContext,<br>
    llm_response: LlmResponse,<br>
) -> LlmResponse:<br>
    del callback_context  # unused<br>
    if not llm_response.content or not llm_response.content.parts:<br>
        return llm_response<br>
    for idx, part in enumerate(llm_response.content.parts):<br>
        if _END_OF_EDIT_MARK in part.text:<br>
            del llm_response.content.parts[idx + 1 :]<br>
            part.text = part.text.split(_END_OF_EDIT_MARK, 1)[0]<br>
    return llm_response<br>
reviser_agent = Agent(<br>
    model=os.getenv("MODEL"),<br>
    name='reviser_agent',<br>
    instruction=prompt.REVISER_PROMPT,<br>
    before_model_callback=log_query_to_model,<br>
    after_model_callback=_remove_end_of_edit_mark,<br>
)